In [ ]:
# Interactively assign a brain region to each recorded channel on the NP2.0
# electrode map (box-select/click select + label from a fixed dropdown), and
# save the result to a per-channel JSON file. Built on Panel + Bokeh — needs
# `pip install panel bokeh` (or conda-forge) in the optonpx env first.
#
# Edit src/region_map.py -> BRAIN_REGIONS before running to match your actual
# target regions (placeholder list ships with the module).
#
# Notebook-only tool: run these cells with #%% in VS Code / Jupyter.

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────────
from pathlib import Path

base_path = Path(r"E:\D1-1-4_IM-1971\ephys_raw\2026-06-19_14-59-14")
PROBE_IDX = 0
# where the assignment is saved / reloaded from — None = default next to continuous.dat
out_path = None

In [ ]:
# ── resolve paths and load probe params ─────────────────────────────────────────
import sys
sys.path.append(str(Path(__file__).parent.parent) if "__file__" in dir() else "..")

from src import oe_parse_folders, oe_parse_params

oe_names, oe_paths = oe_parse_folders(base_path)
probes_params, _   = oe_parse_params(oe_paths["xml"], oe_paths["oebin"])

probe          = probes_params[PROBE_IDX]
continuous_dir = oe_paths["ephys_streams"][PROBE_IDX]

if out_path is None:
    out_path = continuous_dir / "channel_brain_regions.json"

print(f"Recording : {base_path.name}")
print(f"Probe     : {probe['stream_name']}  ({probe['channel_count']} ch)")
print(f"Save path : {out_path}")
if out_path.exists():
    print("Existing assignment found — will preload and resume editing.")
else:
    print("No existing assignment — will start from all-'unknown'.")

In [ ]:
# ── open interactive assignment app ──────────────────────────────────────────────
import panel as pn
pn.extension()

from src import build_region_assignment_app

app, get_assignment = build_region_assignment_app(
    probe["channel_electrode"],
    existing_path=out_path,
    save_path=out_path,
    title=base_path.name,
)
print("\nBox-select / click channels in the app above, then click 'Save'.")
print("Re-run this cell's get_assignment() afterward to inspect the result in-session.")

# explicit display() instead of relying on "last expression in the cell" -- that
# only auto-renders when app truly is the last statement, which broke once the
# print()s above were added after it.
from IPython.display import display
display(app)
# if inline rendering doesn't show up in the VS Code Interactive Window, use instead:
# app.show()   # opens the app in a browser tab via Panel's built-in dev server

In [ ]:
# ── verify the saved file ─────────────────────────────────────────────────────
# Run after clicking "Save" in the app.
from src import load_region_assignment

if not out_path.exists():
    print(f"No file at {out_path} yet — click 'Save' in the app first.")
else:
    saved = load_region_assignment(out_path)
    counts = {}
    for region in saved.values():
        counts[region] = counts.get(region, 0) + 1

    print(f"Loaded {len(saved)} channel assignments from {out_path.name}")
    for region, count in sorted(counts.items()):
        print(f"  {region:20s}: {count}")